In [1]:
import pandas as pd
import pyodbc

# Load the datasets
df_customer_churn = pd.read_csv("BankChurners_clean.csv")
df_customer_churn = df_customer_churn.where(pd.notnull(df_customer_churn), None)
df_customer_churn.head()

,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,...,4,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,1,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000


In [14]:
def connect_sqlserver(server: str, database: str):
    try:
        # Connect to SQL Server
        conn = pyodbc.connect(
            "DRIVER={ODBC Driver 17 for SQL Server};"
            f"SERVER={server};"
            f"DATABASE={database};"
            "Trusted_Connection=yes;"
            "TrustServerCertificate=yes;"
        )
        cursor = conn.cursor()
        cursor.execute("SELECT SYSTEM_USER, DB_NAME()")
        user, db = cursor.fetchone()
        print(f"[✓] Connected successfully! User = {user}, DB = {db}")
        return conn, cursor
    except Exception as e:
        print("[!] Connection failed:", e)
        return None, None

In [ ]:
def insert_dataframe_to_sql(
    df: pd.DataFrame,
    table_name: str,
    columns: list,
    server: str,
    database: str
):
    """
    Insert a pandas DataFrame into SQL Server using Windows Authentication.
    
    Parameters:
        df (pd.DataFrame): The DataFrame to insert.
        table_name (str): Name of the target table, e.g., 'dbo.BankChurners'.
        columns (list): List of columns in the correct SQL order.
        server (str): SQL Server name or hostname.
        database (str): Database name.
    """

    # Convert NaN -> None for SQL NULL compatibility
    df_clean = df.where(pd.notnull(df), None)

    # Build SQL INSERT statement
    placeholders = ", ".join(["?"] * len(columns))
    col_list = ", ".join(columns)
    sql = f"INSERT INTO {table_name} ({col_list}) VALUES ({placeholders})"

    # Convert rows to tuples in correct order
    data_tuples = df_clean[columns].itertuples(index=False, name=None)

    # Connect to SQL Server
    conn, cursor = connect_sqlserver(server=server, database=database)

    if conn:
        # Execute batch insert
        cursor.executemany(sql, data_tuples)
        conn.commit()

        cursor.close()
        conn.close()

    print(f"[✓] Insert completed: {len(df_clean)} rows → {table_name}")

[✓] Connected successfully! User = DESKTOP-05EPS3K\Admin, DB = 03_Credit_Score_DB
[✓] Insert completed: 10127 rows → dbo.BankChurners


In [ ]:
cols = [
    "CLIENTNUM", "Attrition_Flag", "Customer_Age", "Gender", "Dependent_count", "Education_Level",
    "Marital_Status", "Income_Category", "Card_Category", "Months_on_book", "Total_Relationship_Count",
    "Months_Inactive_12_mon", "Contacts_Count_12_mon", "Credit_Limit", "Total_Revolving_Bal",
    "Avg_Open_To_Buy", "Total_Amt_Chng_Q4_Q1", "Total_Trans_Amt", "Total_Trans_Ct",
    "Total_Ct_Chng_Q4_Q1", "Avg_Utilization_Ratio"
]

insert_dataframe_to_sql(
    df=df_customer_churn,
    table_name="dbo.BankChurners",
    columns=cols,
    server="DESKTOP-05EPS3K",
    database="03_Credit_Score_DB"
)